In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}


workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
graph.invoke({"foo": "", "bar":[]}, config)

/Users/sulbhajain/Documents/Personal/genAI_projects/langgraph_agents/lang_env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


{'foo': 'b', 'bar': ['a', 'b']}

In [4]:
# get the latest state snapshot
config = {"configurable": {"thread_id": "1"}}
graph.get_state(config)

# # get a state snapshot for a specific checkpoint_id
# config = {"configurable": {"thread_id": "1", "checkpoint_id": "1ef663ba-28fe-6528-8002-5a559208592c"}}
# graph.get_state(config)

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e504c-9462-6ab2-8002-3858dbd625a6'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-12-29T22:21:56.793410+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e504c-9462-6332-8001-27d4ba0d327b'}}, tasks=(), interrupts=())

In [5]:
config = {"configurable": {"thread_id": "1"}}
list(graph.get_state_history(config))

[StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e504c-9462-6ab2-8002-3858dbd625a6'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-12-29T22:21:56.793410+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e504c-9462-6332-8001-27d4ba0d327b'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e504c-9462-6332-8001-27d4ba0d327b'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-12-29T22:21:56.793216+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e504c-9461-6766-8000-3f1e26f0a8f0'}}, tasks=(PregelTask(id='e4566333-6632-5669-5557-77eb844deca9', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts

In [6]:
config = {"configurable": {"thread_id": "1", "checkpoint_id": "1f0e504c-9462-6332-8001-27d4ba0d327b"}}
graph.invoke(None, config=config)

{'foo': 'b', 'bar': ['a', 'b']}

In [8]:
graph.update_state(config, {"foo": "b", "bar": ["b"]})

KeyError: 'checkpoint_ns'